
#  **Framework-agnostic RAG demo**
This notebook demonstrates the **core primitives** from this lesson without using LangChain or LlamaIndex:
- **Pre-index safety**: redaction + metadata (tenant/roles/users)
- **Chunking** and **embeddings**
- A tiny **index** with vector search (cosine)
- **ACL filtering** via a sidecar table
- **Hybrid retrieval** (vector + BM25-style lexical)
- **Reranking** a shortlist
- **Grounded JSON** output and **verification gates** (Parse → Cite → Support)
- A tiny **micro-eval** (recall@k, p95 retrieval ms, parse-fail %)

> ⚠️ This is *framework-agnostic*. If FAISS / rank_bm25 are available, we’ll use them; otherwise we fall back to pure-Python implementations.



## 1) Imports & utilities
We import only standard scientific Python. If **FAISS** or **rank_bm25** are present, we’ll use them; otherwise we use simple fallbacks.


In [1]:

import math, time, json, re, statistics, random
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import numpy as np
import pandas as pd

# Optional libs (may not be installed in your environment)
try:
    import faiss  # type: ignore
    HAVE_FAISS = True
except Exception:
    HAVE_FAISS = False

try:
    from rank_bm25 import BM25Okapi  # type: ignore
    HAVE_BM25 = True
except Exception:
    HAVE_BM25 = False

random.seed(42)
np.random.seed(42)

print("FAISS available:", HAVE_FAISS)
print("BM25 available:", HAVE_BM25)


FAISS available: True
BM25 available: True



## 2) Sample corpus, redaction & metadata
We’ll create a tiny corpus (5–10 docs), attach minimal **ACL** metadata (**tenant_id**, **allow_roles**, **allow_users**) and run **simple redaction** (mask emails/phones).


In [2]:

def redact(text: str) -> str:
    # Mask emails and phone-like numbers (very simple examples)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[EMAIL]", text)
    text = re.sub(r"\b\+?\d[\d\-\.\s]{7,}\d\b", "[PHONE]", text)
    return text

def join_lines(lines):
    return "\n".join(lines)

docs = [
    {
        "doc_id": "pol_fr_2fa",
        "title": "Security Policy — 2FA (France)",
        "date": "2025-03-01",
        "tenant_id": "acme",
        "allow_roles": ["support","manager"],
        "allow_users": ["u_007"],
        "text": redact(join_lines([
            "For France, 2FA reset requires in-country ID verification.",
            "Contact support-fr@acme.com.",
            "Business hours 9-17 CET.",
            "Escalation path requires ticket code starting with FR-SEC-2FA."
        ])),
        "tags": ["policy","security","france"]
    },
    {
        "doc_id": "faq_2fa_emea",
        "title": "2FA FAQ — EMEA",
        "date": "2025-02-10",
        "tenant_id": "acme",
        "allow_roles": ["support","agent"],
        "allow_users": [],
        "text": redact(join_lines([
            "Resetting 2FA in EMEA: France uses ID verification; Germany requires HR approval.",
            "Two-factor app changes may take 24h to sync.",
            "Phone +33 1 23 45 67 89 is deprecated."
        ])),
        "tags": ["faq","security","emea"]
    },
    {
        "doc_id": "kb_en_onboarding",
        "title": "Onboarding Steps — EN",
        "date": "2024-11-01",
        "tenant_id": "acme",
        "allow_roles": ["hr","manager"],
        "allow_users": [],
        "text": redact(join_lines([
            "New hires: complete tax forms, enroll in benefits, set up 2FA, and review code of conduct.",
            "For issues email onboarding@acme.com.",
            "2FA app: company-approved authenticator."
        ])),
        "tags": ["kb","onboarding"]
    },
    {
        "doc_id": "runbook_us_2fa",
        "title": "Runbook — 2FA (US)",
        "date": "2025-01-05",
        "tenant_id": "acme",
        "allow_roles": ["support","agent"],
        "allow_users": ["u_123"],
        "text": redact(join_lines([
            "US 2FA reset path: verify last 4 digits of SSN and manager approval.",
            "No ID verification required; escalate if account is locked > 48h."
        ])),
        "tags": ["runbook","security","us"]
    },
    {
        "doc_id": "policy_privacy",
        "title": "Privacy Policy Excerpt",
        "date": "2024-12-15",
        "tenant_id": "acme",
        "allow_roles": ["legal","manager"],
        "allow_users": [],
        "text": redact(join_lines([
            "Personal data must not be stored in plain text.",
            "Emails and phone numbers should be masked.",
            "Never store API keys in documents."
        ])),
        "tags": ["policy","privacy"]
    },
]

import pandas as pd
df_docs = pd.DataFrame(docs)
df_docs


,doc_id,title,date,tenant_id,allow_roles,allow_users,text,tags
0,pol_fr_2fa,Security Policy — 2FA (France),2025-03-01,acme,"[support, manager]",[u_007],"For France, 2FA reset requires in-country ID v...","[policy, security, france]"
1,faq_2fa_emea,2FA FAQ — EMEA,2025-02-10,acme,"[support, agent]",[],Resetting 2FA in EMEA: France uses ID verifica...,"[faq, security, emea]"
2,kb_en_onboarding,Onboarding Steps — EN,2024-11-01,acme,"[hr, manager]",[],"New hires: complete tax forms, enroll in benef...","[kb, onboarding]"
3,runbook_us_2fa,Runbook — 2FA (US),2025-01-05,acme,"[support, agent]",[u_123],US 2FA reset path: verify last 4 digits of SSN...,"[runbook, security, us]"
4,policy_privacy,Privacy Policy Excerpt,2024-12-15,acme,"[legal, manager]",[],Personal data must not be stored in plain text...,"[policy, privacy]"



## 3) Chunking (300–800 tokens target; overlap 20–50)
We’ll do a simple **word-count** splitter to approximate tokens. We keep metadata on every chunk.


In [3]:

def chunk_text(text: str, chunk_words=120, overlap_words=20):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+chunk_words]
        if not chunk:
            break
        chunks.append(" ".join(chunk))
        i += chunk_words - overlap_words
    return chunks

rows = []
for d in docs:
    chs = chunk_text(d["text"], chunk_words=80, overlap_words=15)  # small for demo
    for idx, ch in enumerate(chs):
        rows.append({
            "chunk_id": f"{d['doc_id']}__{idx}",
            "doc_id": d["doc_id"],
            "title": d["title"],
            "date": d["date"],
            "tenant_id": d["tenant_id"],
            "allow_roles": d["allow_roles"],
            "allow_users": d["allow_users"],
            "tags": d["tags"],
            "text": ch
        })

df_chunks = pd.DataFrame(rows)
df_chunks.head()


,chunk_id,doc_id,title,date,tenant_id,allow_roles,allow_users,tags,text
0,pol_fr_2fa__0,pol_fr_2fa,Security Policy — 2FA (France),2025-03-01,acme,"[support, manager]",[u_007],"[policy, security, france]","For France, 2FA reset requires in-country ID v..."
1,faq_2fa_emea__0,faq_2fa_emea,2FA FAQ — EMEA,2025-02-10,acme,"[support, agent]",[],"[faq, security, emea]",Resetting 2FA in EMEA: France uses ID verifica...
2,kb_en_onboarding__0,kb_en_onboarding,Onboarding Steps — EN,2024-11-01,acme,"[hr, manager]",[],"[kb, onboarding]","New hires: complete tax forms, enroll in benef..."
3,runbook_us_2fa__0,runbook_us_2fa,Runbook — 2FA (US),2025-01-05,acme,"[support, agent]",[u_123],"[runbook, security, us]",US 2FA reset path: verify last 4 digits of SSN...
4,policy_privacy__0,policy_privacy,Privacy Policy Excerpt,2024-12-15,acme,"[legal, manager]",[],"[policy, privacy]",Personal data must not be stored in plain text...



## 4) Embeddings (hashing) & a tiny "index"
We’ll use a simple **hashing vectorizer** to create fixed-size embeddings (no external models needed).
- Embedding size = 512 dims.
- Similarity = cosine.
- If FAISS is available, we’ll create a FAISS index; otherwise we keep a NumPy matrix.


In [4]:

import numpy as np, re

EMBED_DIM = 512

def hash_vectorize(text: str, dim=EMBED_DIM) -> np.ndarray:
    v = np.zeros(dim, dtype=np.float32)
    for tok in re.findall(r"[A-Za-z0-9_\-]+", text.lower()):
        h = hash(tok) % dim
        v[h] += 1.0
    n = np.linalg.norm(v) + 1e-9
    return v / n

embeddings = np.stack([hash_vectorize(t) for t in df_chunks["text"]])
embeddings.shape


(5, 512)

In [5]:

# Build a "vector index"
try:
    import faiss  # type: ignore
    HAVE_FAISS = True
except Exception:
    HAVE_FAISS = False

if HAVE_FAISS:
    index = faiss.IndexFlatIP(EMBED_DIM)  # cosine with normalized vectors ~ inner product
    index.add(embeddings.astype(np.float32))
    print("FAISS index built:", index.ntotal)
else:
    index = embeddings  # we'll brute-force cosine with NumPy
    print("Using NumPy matrix as index; brute-force cosine.")


FAISS index built: 5



## 5) Lexical scoring (BM25-style)
If `rank_bm25` is present we’ll use it; otherwise a **minimal BM25** fallback is provided.


In [6]:

def tokenize(text: str) -> List[str]:
    return re.findall(r"[A-Za-z0-9_\-]+", text.lower())

try:
    from rank_bm25 import BM25Okapi  # type: ignore
    HAVE_BM25 = True
except Exception:
    HAVE_BM25 = False

if HAVE_BM25:
    tokenized_corpus = [tokenize(t) for t in df_chunks["text"].tolist()]
    bm25 = BM25Okapi(tokenized_corpus)
    def bm25_scores(query: str) -> np.ndarray:
        toks = tokenize(query)
        scores = bm25.get_scores(toks)
        return np.array(scores, dtype=np.float32)
else:
    k1, b = 1.2, 0.75
    tokenized_corpus = [tokenize(t) for t in df_chunks["text"].tolist()]
    N = len(tokenized_corpus)
    df_counts = Counter()
    doc_lens = [len(toks) for toks in tokenized_corpus]
    avgdl = sum(doc_lens)/max(1,len(doc_lens))
    for toks in tokenized_corpus:
        for term in set(toks):
            df_counts[term] += 1

    def idf(term: str) -> float:
        df = df_counts.get(term, 0) + 0.5
        return math.log((N - df + 0.5) / df + 1e-9)

    def bm25_scores(query: str) -> np.ndarray:
        q = tokenize(query)
        scores = np.zeros(N, dtype=np.float32)
        for i, toks in enumerate(tokenized_corpus):
            freq = Counter(toks)
            dl = doc_lens[i]
            s = 0.0
            for term in q:
                tf = freq.get(term, 0)
                if tf == 0: 
                    continue
                denom = tf + k1*(1 - b + b*dl/avgdl)
                s += idf(term) * (tf*(k1+1)) / max(1e-9, denom)
            scores[i] = s
        return scores



## 6) Retrieval with ACL filtering and Hybrid fusion
- **Vector** top-N via cosine (N=50 by default)
- **ACL filter**: tenant/role/user (sidecar inside our chunks table)
- **Hybrid fusion**: combine **vector** and **BM25** scores (weighted sum) → shortlist → keep top-20


In [7]:

def cosine_sim_matrix(vec: np.ndarray, mat: np.ndarray) -> np.ndarray:
    return mat @ vec

def vector_topN(query: str, N=50) -> Tuple[np.ndarray, np.ndarray]:
    qv = hash_vectorize(query)
    if 'faiss' in globals() and isinstance(index, faiss.swigfaiss_avx2.IndexFlatIP):
        D, I = index.search(qv.reshape(1,-1).astype(np.float32), N)
        scores = D[0]; idxs = I[0]
    else:
        sims = cosine_sim_matrix(qv, index)
        idxs = np.argpartition(-sims, min(N, len(sims)-1))[:N]
        scores = sims[idxs]
        order = np.argsort(-scores)
        idxs = idxs[order]; scores = scores[order]
    return idxs, scores

def acl_filter(idxs: np.ndarray, tenant_id: str, role: str, user_id: str) -> np.ndarray:
    keep = []
    for idx in idxs:
        row = df_chunks.iloc[idx]
        if row["tenant_id"] != tenant_id:
            continue
        roles = set(row["allow_roles"] or [])
        users = set(row["allow_users"] or [])
        if (role in roles) or (user_id in users):
            keep.append(idx)
    return np.array(keep, dtype=int)

def hybrid_fusion(vector_scores: np.ndarray, bm25_scores_arr: np.ndarray, alpha=0.7) -> np.ndarray:
    def z(x):
        mu = x.mean() if x.size else 0.0
        sd = x.std() if x.size else 1.0
        return (x - mu) / (sd + 1e-9)
    return alpha * z(vector_scores) + (1 - alpha) * z(bm25_scores_arr)


In [8]:
# --- Robust FAISS / fallback detection ---
import numpy as np

# We created `index` earlier. If it's a NumPy array, we're using the brute-force fallback.
USE_FAISS = "faiss" in globals() and not isinstance(index, np.ndarray)

def vector_topN(query: str, N=50):
    """
    Returns (idxs, scores) for the top-N most similar chunks to the query.
    Uses FAISS if available; otherwise NumPy cosine on the embeddings matrix.
    """
    qv = hash_vectorize(query).astype("float32")  # normalized
    if USE_FAISS:
        # min(N, index.ntotal) avoids asking more than available
        D, I = index.search(qv.reshape(1, -1), min(N, getattr(index, "ntotal", len(embeddings))))
        idxs = I[0]
        scores = D[0]
        return idxs, scores
    else:
        # Brute-force cosine with our normalized embeddings
        sims = embeddings @ qv  # (N,) because both are L2-normalized
        n = min(N, len(sims))
        idxs = np.argpartition(-sims, n-1)[:n]
        order = np.argsort(-sims[idxs])
        idxs = idxs[order]
        scores = sims[idxs]
        return idxs, scores

print("USE_FAISS:", USE_FAISS, "| index type:", type(index))

USE_FAISS: True | index type: <class 'faiss.swigfaiss.IndexFlatIP'>


In [9]:

def retrieve(query: str, tenant_id="acme", role="support", user_id="u_007",
             N=50, shortlist=20, alpha=0.7):
    idxs, vec_scores = vector_topN(query, N=N)
    idxs = acl_filter(idxs, tenant_id=tenant_id, role=role, user_id=user_id)
    if idxs.size == 0:
        return np.array([], dtype=int), np.array([])
    bm_scores_full = bm25_scores(query)
    bm_scores = bm_scores_full[idxs]
    fused = hybrid_fusion(vec_scores[:len(idxs)], bm_scores, alpha=alpha)
    order = np.argsort(-fused)[:shortlist]
    return idxs[order], fused[order]

q = "How do I reset 2FA in France?"
idxs, scores = retrieve(q, tenant_id="acme", role="support", user_id="u_007", N=50, shortlist=20, alpha=0.7)
df_chunks.iloc[idxs][["chunk_id","doc_id","title","text"]]


,chunk_id,doc_id,title,text
0,pol_fr_2fa__0,pol_fr_2fa,Security Policy — 2FA (France),"For France, 2FA reset requires in-country ID v..."
1,faq_2fa_emea__0,faq_2fa_emea,2FA FAQ — EMEA,Resetting 2FA in EMEA: France uses ID verifica...
3,runbook_us_2fa__0,runbook_us_2fa,Runbook — 2FA (US),US 2FA reset path: verify last 4 digits of SSN...



## 7) Reranking a shortlist → **top-k**
We’ll use a **surrogate reranker**: measure similarity between the **query** and each candidate chunk using a denser lexical match (BM25) plus cosine on hashing-embeds.
Then keep **top-k** (k=3–5).


In [10]:

def surrogate_rerank(query: str, candidate_idxs: np.ndarray, k=5, beta=0.5):
    qv = hash_vectorize(query)
    vec_scores = (embeddings[candidate_idxs] @ qv)
    bm_full = bm25_scores(query)
    bm = bm_full[candidate_idxs]
    def z(x):
        mu, sd = float(np.mean(x)), float(np.std(x) + 1e-9)
        return (x - mu) / sd
    fused = beta * z(bm) + (1 - beta) * z(vec_scores)
    order = np.argsort(-fused)[:k]
    return candidate_idxs[order], fused[order]

topk = 5
reranked_idxs, reranked_scores = surrogate_rerank(q, idxs, k=topk, beta=0.6)
df_chunks.iloc[reranked_idxs][["chunk_id","doc_id","title","text"]]


,chunk_id,doc_id,title,text
0,pol_fr_2fa__0,pol_fr_2fa,Security Policy — 2FA (France),"For France, 2FA reset requires in-country ID v..."
1,faq_2fa_emea__0,faq_2fa_emea,2FA FAQ — EMEA,Resetting 2FA in EMEA: France uses ID verifica...
3,runbook_us_2fa__0,runbook_us_2fa,Runbook — 2FA (US),US 2FA reset path: verify last 4 digits of SSN...



## 8) Grounded prompt, JSON contract & verification gates
We’ll build a tiny **prompt skeleton** and produce a **mock JSON** answer. Then we enforce:
- **Parse**: JSON is valid and schema matches
- **Cite**: every claim cites at least one provided chunk id
- **Support**: key terms in each sentence appear in the cited chunk(s)


In [11]:

SYSTEM_RULES = (
    "Use ONLY the provided CONTEXT. If insufficient, say so and return sources=[]. "
    "Return valid JSON: { \"answer\": string, \"sources\": string[] }. "
    "Cite chunk ids for each factual claim."
)

def build_context_block(idxs: np.ndarray) -> str:
    parts = []
    for idx in idxs:
        row = df_chunks.iloc[idx]
        hdr = f"[{row['chunk_id']}] Title: {row['title']}"
        parts.append(hdr + "\n" + row["text"])
    return "\n\n".join(parts)

def mock_llm_answer(query: str, topk_idxs: np.ndarray) -> str:
    if topk_idxs.size == 0:
        return json.dumps({"answer":"I don't have enough information from the provided sources.", "sources":[]})
    first = df_chunks.iloc[topk_idxs[0]]["text"]
    sent = first.split(".")[0].strip()
    cited = [df_chunks.iloc[i]["chunk_id"] for i in topk_idxs]
    return json.dumps({"answer": sent, "sources": cited})

def parse_gate(raw_json: str) -> Tuple[bool, Dict[str, Any]]:
    try:
        obj = json.loads(raw_json)
        ok = isinstance(obj, dict) and "answer" in obj and "sources" in obj and isinstance(obj["sources"], list)
        return ok, obj if ok else {}
    except Exception:
        return False, {}

def cite_gate(obj: Dict[str, Any], provided_idxs: np.ndarray) -> bool:
    provided_ids = {df_chunks.iloc[i]["chunk_id"] for i in provided_idxs}
    cited = set(obj.get("sources", []))
    return len(cited) >= 1 and cited.issubset(provided_ids)

def tokenize_simple(s: str):
    return re.findall(r"[A-Za-z0-9_\-]+", s.lower())

def support_gate(obj: Dict[str, Any]) -> bool:
    answer = obj.get("answer","")
    cited = obj.get("sources", [])
    if not cited:
        return False
    sents = [s.strip() for s in re.split(r"[.!?]+", answer) if s.strip()]
    if not sents:
        return False
    for s in sents:
        toks = set(tokenize_simple(s)) - {"the","a","an","in","for","to","of","and","or","with","on","is","are"}
        if not toks:
            continue
        ok_any = False
        for cid in cited:
            row = df_chunks[df_chunks["chunk_id"]==cid].iloc[0]
            chunk_toks = set(tokenize_simple(row["text"]))
            if len(toks & chunk_toks) >= max(1, len(toks)//6):
                ok_any = True; break
        if not ok_any:
            return False
    return True

context_block = build_context_block(reranked_idxs)
prompt = f"SYSTEM:\n{SYSTEM_RULES}\n\nUSER:\n{q}\n\nCONTEXT:\n{context_block}"
raw = mock_llm_answer(q, reranked_idxs)

ok_parse, obj = parse_gate(raw)
ok_cite = cite_gate(obj, reranked_idxs) if ok_parse else False
ok_support = support_gate(obj) if ok_cite else False

print("JSON:", raw)
print("Parse ✓:", ok_parse, " | Cite ✓:", ok_cite, " | Support ✓:", ok_support)


JSON: {"answer": "For France, 2FA reset requires in-country ID verification", "sources": ["pol_fr_2fa__0", "faq_2fa_emea__0", "runbook_us_2fa__0"]}
Parse ✓: True  | Cite ✓: True  | Support ✓: True



## 9) Micro-eval: recall@k, p95 retrieval ms, parse-fail %
We’ll define a small canary set (query → expected doc_id) and measure:
- **recall@k** (does expected doc appear in top-k?)
- **p95 retrieval** (vector+hybrid+rerank, excluding LLM)
- **parse-fail %** (how often JSON fails to parse)


In [12]:

canary = [
    {"q":"How do I reset 2FA in France?", "expected_doc":"pol_fr_2fa"},
    {"q":"What is the 2FA path in the US?", "expected_doc":"runbook_us_2fa"},
    {"q":"Do we store emails in plain text?", "expected_doc":"policy_privacy"},
]

def eval_once(q: str, expected_doc: str, k=5):
    t0 = time.time()
    idxs, _ = retrieve(q, tenant_id="acme", role="support", user_id="u_007", N=50, shortlist=20, alpha=0.7)
    rer_idxs, _ = surrogate_rerank(q, idxs, k=k, beta=0.6) if idxs.size else (np.array([],dtype=int), np.array([]))
    t1 = time.time()
    doc_ids = set(df_chunks.iloc[rer_idxs]["doc_id"].tolist())
    recall = 1.0 if expected_doc in doc_ids else 0.0
    raw = mock_llm_answer(q, rer_idxs)
    ok_parse, obj = parse_gate(raw)
    return (recall, (t1-t0)*1000.0, 0 if ok_parse else 1)

results = [eval_once(x["q"], x["expected_doc"], k=5) for x in canary]
recalls = [r[0] for r in results]
latencies = [r[1] for r in results]
parse_fails = [r[2] for r in results]

print("recall@5 (avg):", sum(recalls)/len(recalls))
print("p95 retrieval ms:", np.percentile(latencies, 95))
print("parse-fail %:", 100*sum(parse_fails)/len(parse_fails))

import pandas as pd
pd.DataFrame({"query":[x["q"] for x in canary],
              "recall@5":recalls,
              "retrieval_ms":latencies,
              "parse_fail":parse_fails})


recall@5 (avg): 0.6666666666666666
p95 retrieval ms: 2.076125144958496
parse-fail %: 0.0


,query,recall@5,retrieval_ms,parse_fail
0,How do I reset 2FA in France?,1.0,1.960039,0
1,What is the 2FA path in the US?,1.0,2.089024,0
2,Do we store emails in plain text?,0.0,0.960827,0



## 10) Appendix — knobs to try
- **Chunking**: increase chunk size to 300–600 words, overlap 20–50, and observe retrieval quality.
- **ACL**: switch `role="support"` to another role and confirm ACL filtering.
- **Hybrid**: change `alpha` (e.g., 0.5) in `retrieve()` to weight keyword vs vector.
- **Shortlist & rerank**: use `shortlist=50` and `k=3` to see if ordering improves.
- **Search depth**: if you add FAISS/HNSW later, experiment with deeper/shallower search params.
- **Verifier**: strengthen `support_gate()` to check numbers/dates more strictly.
